# Phase 1 & 2: Preprocessing and Classical NLP (TF-IDF + Logistic Regression / Naive Bayes)

This notebook covers the preprocessing of Twitter text data and the implementation of classical statistical models for sentiment analysis on the **BrandPulse AI** platform.

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# add parent dir to path so we can import our preprocessing script
import sys
sys.path.append('..')
from preprocessing import clean_tweet

print("Imports completed successfully.")

## 1. Data Collection and Loading

We are downloading the **Twitter US Airline Sentiment** dataset from a public GitHub repository.

In [ ]:
# Create necessary folders
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

data_path = '../data/Tweets.csv'
urls = [
    'https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv',
    'https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/refs/heads/master/Tweets.csv',
    'https://raw.githubusercontent.com/kolaveridi/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv'
]

if not os.path.exists(data_path):
    print("Downloading dataset...")
    success = False
    for url in urls:
        try:
            print(f"Trying to download from: {url}")
            urllib.request.urlretrieve(url, data_path)
            print("Download finished!")
            success = True
            break
        except Exception as e:
            print(f"Failed: {e}")
    if not success:
        raise RuntimeError("Could not download the dataset from any source.")
else:
    print("Dataset already exists.")

In [ ]:
# Read the CSV
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print("\nData preview:")
print(df[['airline_sentiment', 'text']].head())

## 2. Linguistic Preprocessing (Cleaning Tweets)

We apply the functions defined in `preprocessing.py` to clean each tweet (lowercase, remove mentions/links, lemmatize, and remove stop words).

In [ ]:
print("Cleaning tweets (this might take a minute)...")
df['clean_text'] = df['text'].apply(clean_tweet)

# drop empty rows
df = df[df['clean_text'].str.strip() != '']
print("Cleaning finished!")
print(df[['airline_sentiment', 'clean_text']].head())

## 3. Exploratory Data Analysis (EDA)

Let's see how the sentiments are distributed and which words appear most frequently.

In [ ]:
# Sentiment distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='airline_sentiment', data=df, order=['positive', 'neutral', 'negative'], palette='viridis')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Tweet Count')
plt.savefig('../data/sentiment_distribution.png', bbox_inches='tight')
plt.show()

print(df['airline_sentiment'].value_counts(normalize=True))

In [ ]:
# Generate Word Clouds for each sentiment
sentiments = ['positive', 'neutral', 'negative']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, sent in enumerate(sentiments):
    text = " ".join(df[df['airline_sentiment'] == sent]['clean_text'])
    wordcloud = WordCloud(width=800, height=800, background_color='white', max_words=80).generate(text)
    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].set_title(f"Sentiment: {sent.capitalize()}", fontsize=16)
    axes[i].axis('off')

plt.suptitle('Word Clouds by Sentiment', fontsize=20)
plt.savefig('../data/wordclouds.png', bbox_inches='tight')
plt.show()

## 4. TF-IDF Vectorization & Data Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['airline_sentiment']

# 80/20 train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 5. Training Classical Models

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

# 1. Logistic Regression
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

print("Training Logistic Regression...")
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("\nClassification Report: Logistic Regression")
print(classification_report(y_test, y_pred_lr))

In [ ]:
# 2. Naive Bayes
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf', MultinomialNB())
])

print("Training Naive Bayes...")
nb_pipeline.fit(X_train, y_train)
y_pred_nb = nb_pipeline.predict(X_test)

print("\nClassification Report: Naive Bayes")
print(classification_report(y_test, y_pred_nb))

## 6. Results & Confusion Matrices

In [ ]:
# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_nb, ax=axes[1], cmap='Greens')
axes[1].set_title('Naive Bayes')

plt.tight_layout()
plt.savefig('../data/confusion_matrices_classical.png', bbox_inches='tight')
plt.show()

## 7. Saving the Best Model

In [ ]:
# Logistic Regression performs well enough, so we'll save it to disk
joblib.dump(lr_pipeline, '../models/classical_pipeline.pkl')
print("Modèle Logistic Regression + TF-IDF sauvegardé dans '../models/classical_pipeline.pkl'.")